# Install Library

In [43]:
!pip -q install pymupdf
!pip -q install sentence-transformers
!pip -q install scikit-learn
!pip -q install pandas
!pip -q install tqdm
!pip -q install Sastrawi
!pip -q install nltk
!pip -q install sentence-transformers

# Import Library

In [44]:
import os
import re
import fitz
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Upload Data

In [67]:
!git clone https://github.com/didan-ui/Case-Based-Reasoning.git
%cd Case-Based-Reasoning

Cloning into 'Case-Based-Reasoning'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 74 (delta 5), reused 22 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 12.48 MiB | 32.61 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/Case-Based-Reasoning/Case-Based-Reasoning


In [68]:
raw_path = "data/raw"

pdf_files = sorted([
    f for f in os.listdir(raw_path)
    if f.lower().endswith(".pdf")
])

print(f"Jumlah PDF : {len(pdf_files)}")
print(pdf_files)

Jumlah PDF : 50
['putusan_10_pid.sus_2024_pn_yyk_20260711114334.pdf', 'putusan_121_pid.sus_2021_pn_yyk_20260711105105.pdf', 'putusan_126_pid.sus_2021_pn_yyk_20260711105054.pdf', 'putusan_134_pid.sus_2021_pn_yyk_20260711105120.pdf', 'putusan_135_pid.sus_2021_pn_yyk_20260711105126.pdf', 'putusan_136_pid.sus_2021_pn_yyk_20260711105109.pdf', 'putusan_142_pid.sus_2021_pn_yyk_20260711105114.pdf', 'putusan_145_pid.sus_2021_pn_yyk_20260711104555.pdf', 'putusan_152_pid.sus_2021_pn_yyk_20260711105051.pdf', 'putusan_153_pid.sus_2021_pn_yyk_20260711105046.pdf', 'putusan_154_pid.sus_2021_pn_yyk_20260711105035.pdf', 'putusan_155_pid.sus_2021_pn_yyk_20260711104531.pdf', 'putusan_157_pid.sus_2021_pn_yyk_20260711104458.pdf', 'putusan_163_pid.sus_2021_pn_yyk_20260711104836.pdf', 'putusan_164_pid.sus_2021_pn_yyk_20260711104607.pdf', 'putusan_165_pid.sus_2021_pn_yyk_20260711104519.pdf', 'putusan_169_pid.sus_2020_pn_yyk_20260711113713.pdf', 'putusan_170_pid.sus_2021_pn_yyk_20260711104543.pdf', 'putusan_175

In [69]:
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/eval", exist_ok=True)
os.makedirs("data/results", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# TAHAP 1: MEMBANGUN CASE BASE
**Deskripsi:** Melakukan akuisisi data (PDF), ekstraksi teks, pembersihan data (cleaning), dan validasi keutuhan teks.

# Cleaning

In [70]:
def clean_text(text):
    text = text.lower()

    # hapus disclaimer panjang
    text = re.sub(
        r'disclaimer kepaniteraan mahkamah agung republik indonesia.*?halaman\s+\d+',
        ' ',
        text,
        flags=re.DOTALL
    )

    # hapus header MA
    patterns = [
        r'direktori putusan mahkamah agung republik indonesia',
        r'putusan\.mahkamahagung\.go\.id',
        r'ahkamah agung republik indonesia',
        r'mah agung republik indonesia',
        r'blik indonesia',
        r'hkama',
        r'repub',
    ]

    for p in patterns:
        text = re.sub(p, ' ', text)

    # hapus nomor halaman
    text = re.sub(r'halaman\s+\d+\s+dari\s+\d+\s+halaman', ' ', text)
    text = re.sub(r'halaman\s+\d+', ' ', text)

    # rapikan spasi
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Ekstrak Semua PDF

In [71]:
processed_path = "data/processed"

for i, pdf_file in enumerate(tqdm(pdf_files), start=1):
    pdf_path = os.path.join(raw_path, pdf_file)

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    text = clean_text(text)

    output_path = os.path.join(
        processed_path,
        f"case_{i:03d}.txt"
    )

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)

print("✅ Semua PDF berhasil diekstrak.")

100%|██████████| 50/50 [00:05<00:00,  8.65it/s]

✅ Semua PDF berhasil diekstrak.


# Validasi

In [72]:
txt_files = sorted(os.listdir(processed_path))

print(f"Jumlah TXT : {len(txt_files)}")

for file in txt_files:
    print(file)

Jumlah TXT : 50
case_001.txt
case_002.txt
case_003.txt
case_004.txt
case_005.txt
case_006.txt
case_007.txt
case_008.txt
case_009.txt
case_010.txt
case_011.txt
case_012.txt
case_013.txt
case_014.txt
case_015.txt
case_016.txt
case_017.txt
case_018.txt
case_019.txt
case_020.txt
case_021.txt
case_022.txt
case_023.txt
case_024.txt
case_025.txt
case_026.txt
case_027.txt
case_028.txt
case_029.txt
case_030.txt
case_031.txt
case_032.txt
case_033.txt
case_034.txt
case_035.txt
case_036.txt
case_037.txt
case_038.txt
case_039.txt
case_040.txt
case_041.txt
case_042.txt
case_043.txt
case_044.txt
case_045.txt
case_046.txt
case_047.txt
case_048.txt
case_049.txt
case_050.txt


# Cek Isi Salah Satu File

In [73]:
sample_file = os.path.join(processed_path, txt_files[0])

with open(sample_file, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:3000])

a h agung blik indonesi dari 40 putusan nomor 10/pid.sus/2024/pn yyk p u t u s a n nomor 10/pid.sus/2024/pn yyk demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri yogyakarta yang mengadili perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai berikut dalam perkara terdakwa : 1. nama lengkap : muhammad faris isra hatala bin ridwan hatala (alm) 2. tempat lahir : jayapura 3. umur/tanggal lahir : 22/15 oktober 2001 4. jenis kelamin : laki-laki 5. kebangsaan : indonesia 6. tempat tinggal : jalan mambruk t/rw 002/000 kelurahan wagom kecamatan pariwari kabupaten fak fak provinsi papua barat (sesuai kartu tanda penduduk) atau griya joho asri 4 desa joho kecamatan prambanan kabupaten klaten provinsi jawa tengah (alamat tinggal) 7. agama : islam 8. pekerjaan : pelajar/mahasiswa terdakwa ditangkap berdasarkan surat perintah penangkapan nomor : sp.kap/149/x/res.42/2023/satresnarkoba tertanggal 31 oktober 2023, sejak tanggal 31 oktober 2023

# TAHAP 2: CASE REPRESENTATION
**Deskripsi:** Mentransformasi teks yang sudah dibersihkan ke dalam format terstruktur (JSON/CSV) dengan mengekstrak fitur kunci seperti nomor putusan, fakta hukum, dan amar.

## Ekstraksi Metadata

In [74]:
def extract_metadata(text):
    metadata = {}

    # Nomor perkara
    nomor = re.search(
        r'putusan\s+nomor\s+([0-9]+/[a-z.\-]+/[0-9]+/pn\s*[a-z]+)',
        text,
        re.IGNORECASE
    )
    metadata["nomor_perkara"] = nomor.group(1).upper() if nomor else ""

    # Nama terdakwa
    nama = re.search(
        r'nama lengkap\s*:\s*(.*?)\s*2\.\s*tempat lahir',
        text,
        re.IGNORECASE | re.DOTALL
    )
    metadata["nama_terdakwa"] = nama.group(1).replace(";", "").strip() if nama else ""

    # Pengadilan
    pengadilan = re.search(
        r'pengadilan negeri\s+([a-z ]+?)\s+yang mengadili',
        text,
        re.IGNORECASE
    )
    metadata["pengadilan"] = (
        "Pengadilan Negeri " + pengadilan.group(1).title().strip()
        if pengadilan else ""
    )

    # Pasal (ambil kemunculan pertama)
    pasal = re.search(
        r'pasal\s+[0-9]+[a-z]?(?:\s+ayat\s*\(\d+\))?',
        text,
        re.IGNORECASE
    )
    metadata["pasal"] = pasal.group(0).title() if pasal else ""

    # Amar putusan
    amar = re.search(
        r'mengadili\s*:(.*?)(?:demikian diputuskan|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )
    metadata["amar_putusan"] = amar.group(1).strip() if amar else ""

    return metadata

# Membuat cases.csv

In [75]:
records = []

txt_files = sorted(os.listdir("data/processed"))

for i, file in enumerate(txt_files, start=1):

    with open(os.path.join("data/processed", file), "r", encoding="utf-8") as f:
        text = f.read()

    meta = extract_metadata(text)

    records.append({
      "case_id": f"case_{i:03d}",
      "nomor_perkara": meta["nomor_perkara"],
      "nama_terdakwa": meta["nama_terdakwa"],
      "pengadilan": meta["pengadilan"],
      "pasal": meta["pasal"],
      "amar_putusan": meta["amar_putusan"],
      "text_full": text
    })

cases_df = pd.DataFrame(records)

cases_df.to_csv("data/processed/cases.csv", index=False)

cases_df.head()

,case_id,nomor_perkara,nama_terdakwa,pengadilan,pasal,amar_putusan,text_full
0,case_001,10/PID.SUS/2024/PN YYK,muhammad faris isra hatala bin ridwan hatala (...,Pengadilan Negeri Yogyakarta,Pasal 111 Ayat (1),1. menyatakan terdakwa muhammad faris isra hat...,a h agung blik indonesi dari 40 putusan nomor ...
1,case_002,,,Pengadilan Negeri Yogyakarta,Pasal 127 Ayat (1),,a h agung blik indonesi p u t u s a n nomor : ...
2,case_003,,,Pengadilan Negeri Yogyakarta,Pasal 127,,a h agung blik indonesi p u t u s a n nomor : ...
3,case_004,,,Pengadilan Negeri Yogyakarta,Pasal 14 Ayat (3),,a h agung blik indonesi p u t u s a n nomor 13...
4,case_005,,,Pengadilan Negeri Yogyakarta,Pasal 62,,a h agung blik indonesi p u t u s a n nomor 13...


# Simpan ke JSON

In [76]:
cases_df.to_json(
    "data/processed/cases.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("cases.csv dan cases.json berhasil dibuat.")

cases.csv dan cases.json berhasil dibuat.


# Membuat case_description

In [78]:
def extract_case_description(text):
    """
    Mengambil bagian deskripsi kasus
    mulai dari identitas terdakwa
    sampai sebelum bagian 'mengadili'
    """

    match = re.search(
        r'nama lengkap\s*:.*?(?=mengadili\s*:)',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(0).strip()

    return text

# Tambahkan ke DataFrame

In [79]:
cases_df["case_description"] = cases_df["text_full"].apply(extract_case_description)

cases_df.head()

,case_id,nomor_perkara,nama_terdakwa,pengadilan,pasal,amar_putusan,text_full,case_description
0,case_001,10/PID.SUS/2024/PN YYK,muhammad faris isra hatala bin ridwan hatala (...,Pengadilan Negeri Yogyakarta,Pasal 111 Ayat (1),1. menyatakan terdakwa muhammad faris isra hat...,a h agung blik indonesi dari 40 putusan nomor ...,nama lengkap : muhammad faris isra hatala bin ...
1,case_002,,,Pengadilan Negeri Yogyakarta,Pasal 127 Ayat (1),,a h agung blik indonesi p u t u s a n nomor : ...,a h agung blik indonesi p u t u s a n nomor : ...
2,case_003,,,Pengadilan Negeri Yogyakarta,Pasal 127,,a h agung blik indonesi p u t u s a n nomor : ...,a h agung blik indonesi p u t u s a n nomor : ...
3,case_004,,,Pengadilan Negeri Yogyakarta,Pasal 14 Ayat (3),,a h agung blik indonesi p u t u s a n nomor 13...,a h agung blik indonesi p u t u s a n nomor 13...
4,case_005,,,Pengadilan Negeri Yogyakarta,Pasal 62,,a h agung blik indonesi p u t u s a n nomor 13...,a h agung blik indonesi p u t u s a n nomor 13...


# Atur Urutan Kolom

In [80]:
cases_df = cases_df[
    [
        "case_id",
        "nomor_perkara",
        "nama_terdakwa",
        "pengadilan",
        "pasal",
        "case_description",
        "amar_putusan",
        "text_full"
    ]
]

cases_df.head()

,case_id,nomor_perkara,nama_terdakwa,pengadilan,pasal,case_description,amar_putusan,text_full
0,case_001,10/PID.SUS/2024/PN YYK,muhammad faris isra hatala bin ridwan hatala (...,Pengadilan Negeri Yogyakarta,Pasal 111 Ayat (1),nama lengkap : muhammad faris isra hatala bin ...,1. menyatakan terdakwa muhammad faris isra hat...,a h agung blik indonesi dari 40 putusan nomor ...
1,case_002,,,Pengadilan Negeri Yogyakarta,Pasal 127 Ayat (1),a h agung blik indonesi p u t u s a n nomor : ...,,a h agung blik indonesi p u t u s a n nomor : ...
2,case_003,,,Pengadilan Negeri Yogyakarta,Pasal 127,a h agung blik indonesi p u t u s a n nomor : ...,,a h agung blik indonesi p u t u s a n nomor : ...
3,case_004,,,Pengadilan Negeri Yogyakarta,Pasal 14 Ayat (3),a h agung blik indonesi p u t u s a n nomor 13...,,a h agung blik indonesi p u t u s a n nomor 13...
4,case_005,,,Pengadilan Negeri Yogyakarta,Pasal 62,a h agung blik indonesi p u t u s a n nomor 13...,,a h agung blik indonesi p u t u s a n nomor 13...


# Save Dataframe

In [81]:
cases_df.to_csv(
    "data/processed/cases.csv",
    index=False
)

cases_df.to_json(
    "data/processed/cases.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("✅ cases.csv berhasil diperbarui")

✅ cases.csv berhasil diperbarui


# TAHAP 3: CASE RETRIEVAL
**Deskripsi:** Mengimplementasikan pencarian kemiripan kasus menggunakan metode seperti TF-IDF dan Cosine Similarity untuk menemukan kasus terdahulu yang relevan.

# Preprocessing

In [82]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_words = set(stopwords.words("indonesian"))


def preprocess(text):

    text = text.lower()

    # hapus karakter selain huruf
    text = re.sub(r'[^a-z\s]', ' ', text)

    words = []

    for word in text.split():

        if word not in stop_words:
            words.append(stemmer.stem(word))

    return " ".join(words)

# Preprocess Semua Case Description

In [83]:
cases_df["processed_text"] = cases_df["case_description"].apply(preprocess)

cases_df[
    ["case_id","processed_text"]
].head()

,case_id,processed_text
0,case_001,nama lengkap muhammad faris isra hatala bin ri...
1,case_002,a h agung blik indonesi p u t u s a n nomor pi...
2,case_003,a h agung blik indonesi p u t u s a n nomor pi...
3,case_004,a h agung blik indonesi p u t u s a n nomor pi...
4,case_005,a h agung blik indonesi p u t u s a n nomor pi...


# TF-IDF Vectorizer

In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    cases_df["processed_text"]
)

print(tfidf_matrix.shape)

(50, 4111)


# Fungsi Retrieval TF-IDF

In [98]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_tfidf(query, top_k=5):

    query = preprocess(query)

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_index = similarity.argsort()[::-1][:top_k]

    result = cases_df.iloc[top_index].copy()

    result["similarity"] = similarity[top_index]

    return result[
      [
          "case_id",
          "nomor_perkara",
          "nama_terdakwa",
          "pengadilan",
          "pasal",
          "similarity"
      ]
    ]

# Uji Coba

In [86]:
query = "penyalahgunaan narkotika golongan 1"

retrieve_tfidf(query)

,case_id,nomor_perkara,pasal,similarity
27,case_028,301/PID.SUS/2019/PN YYK,Pasal 127 Ayat (1),0.157959
40,case_041,381/PID.SUS/2025/PN YYK,Pasal 127,0.144032
14,case_015,,Pasal 127 Ayat (1),0.120472
13,case_014,,Pasal 127 Ayat (1),0.117082
31,case_032,324/PID.SUS/2025/PN YYK,Pasal 127 Ayat (1),0.113415


# Simpan Hasil Preprocessing

In [87]:
query = "penyalahgunaan narkotika"

result = retrieve_tfidf(query)

result.to_csv(
    "data/results/tfidf_results.csv",
    index=False
)

result

,case_id,nomor_perkara,pasal,similarity
27,case_028,301/PID.SUS/2019/PN YYK,Pasal 127 Ayat (1),0.157959
40,case_041,381/PID.SUS/2025/PN YYK,Pasal 127,0.144032
14,case_015,,Pasal 127 Ayat (1),0.120472
13,case_014,,Pasal 127 Ayat (1),0.117082
31,case_032,324/PID.SUS/2025/PN YYK,Pasal 127 Ayat (1),0.113415


# Load IndoBERT

In [88]:
model = SentenceTransformer(
    "firqaaa/indo-sentence-bert-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Encode Semua Dokumen

In [89]:
document_embeddings = model.encode(
    cases_df["case_description"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

# Retrieval

In [99]:
def retrieve_indobert(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    similarity = cosine_similarity(
        query_embedding,
        document_embeddings
    ).flatten()

    top_index = similarity.argsort()[::-1][:top_k]

    result = cases_df.iloc[top_index].copy()

    result["similarity"] = similarity[top_index]

    return result[
      [
          "case_id",
          "nomor_perkara",
          "nama_terdakwa",
          "pengadilan",
          "pasal",
          "similarity"
      ]
  ]

# Test

In [92]:
query = "penyalahgunaan narkotika golongan 1"

retrieve_indobert(query)

,case_id,nomor_perkara,pasal,similarity
18,case_019,175/PID.SUS/2020/PN YYK,Pasal 127 Ayat (1),0.210056
19,case_020,176/PID.SUS/2020/PN YYK,Pasal 127 Ayat (1),0.205952
27,case_028,301/PID.SUS/2019/PN YYK,Pasal 127 Ayat (1),0.204586
14,case_015,,Pasal 127 Ayat (1),0.199810
43,case_044,418/PID.SUS/2025/PN YYK,Pasal 436 Ayat (2),0.197488


# TAHAP 4: REUSE
**Deskripsi:** Menghasilkan prediksi solusi (amar putusan) dari query baru berdasarkan Top-K kasus yang ditemukan.

# Fungsi Weighted Similarity

In [93]:
from collections import defaultdict

def weighted_similarity(result_df):

    scores = defaultdict(float)

    for _, row in result_df.iterrows():
        scores[row["pasal"]] += row["similarity"]

    predicted_pasal = max(scores, key=scores.get)

    return predicted_pasal

# Fungsi Reuse

In [94]:
def reuse_solution(query, method="tfidf", top_k=5):

    # Retrieval
    if method == "tfidf":
        retrieved = retrieve_tfidf(query, top_k)
    else:
        retrieved = retrieve_indobert(query, top_k)

    # Prediksi pasal
    predicted_pasal = weighted_similarity(retrieved)

    # Ambil kasus paling mirip
    top_case = retrieved.iloc[0]

    # Ambil amar putusan dari top case
    solution = cases_df.loc[
        cases_df["case_id"] == top_case["case_id"],
        "amar_putusan"
    ].values[0]

    return {
        "query": query,
        "predicted_pasal": predicted_pasal,
        "recommended_case": top_case["case_id"],
        "similarity": top_case["similarity"],
        "amar_putusan": solution,
        "top5": retrieved
    }

# Uji Coba

In [95]:
result = reuse_solution(
    "penyalahgunaan narkotika golongan 1",
    method="tfidf"
)

# Tampilkan Hasil

In [96]:
print("="*60)
print("QUERY")
print(result["query"])

print("\nPREDIKSI PASAL")
print(result["predicted_pasal"])

print("\nKASUS PALING MIRIP")
print(result["recommended_case"])

print("\nSIMILARITY")
print(round(result["similarity"],4))

print("\nAMAR PUTUSAN")
print(result["amar_putusan"])

QUERY
penyalahgunaan narkotika golongan 1

PREDIKSI PASAL
Pasal 127 Ayat (1)

KASUS PALING MIRIP
case_028

SIMILARITY
0.158

AMAR PUTUSAN
1. menyatakan terdakwa rian danu kusuma dewa tersebut diatas, terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana “penyalahgunaan narkotika golongan i bagi diri sendiri” sebagaimana dalam dakwaan kedua; 2. menjatuhkan pidana kepada terdakwa oleh karena itu dengan pidana penjara selama 7 (tujuh) bulan; 3. menetapkan masa penangkapan dan penahanan yang telah dijalani terdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan; 4. menetapkan terdakwa tetap ditahan; 5. menetapkan barang bukti berupa: - 1 (satu) buah tas warna hitam isi 1 (satu) bungkus plastik klip isi tembakau gorilla; - 1 (satu) bekas bungkus rokok gudang garam surya, yang di dalamnya terdapat 4 (empat) linting rokok tembakau gorilla; - 2 (dua) buah paper; - 1 (satu) bungkus tembakau rokok merek violin; disclaimer kepaniteraan m berusaha untuk selalu mencantumkan inform

# Simpan ke CSV

In [97]:
prediction_df = pd.DataFrame({
    "query": [result["query"]],
    "predicted_pasal": [result["predicted_pasal"]],
    "recommended_case": [result["recommended_case"]],
    "similarity": [result["similarity"]],
    "amar_putusan": [result["amar_putusan"]]
})

prediction_df.to_csv(
    "data/results/predictions.csv",
    index=False
)

prediction_df

,query,predicted_pasal,recommended_case,similarity,amar_putusan
0,penyalahgunaan narkotika golongan 1,Pasal 127 Ayat (1),case_028,0.157959,1. menyatakan terdakwa rian danu kusuma dewa t...


# TAHAP 5: Evaluation
**Deskripsi:** Mengevaluasi kinerja model.

# Generate Evaluation Query

In [101]:
import re

def generate_query(text):

    markers = [
        "menimbang bahwa",
        "bahwa terdakwa",
        "bahwa pada",
        "berawal ketika"
    ]

    text = text.lower()

    start = 0

    for marker in markers:
        idx = text.find(marker)
        if idx != -1:
            start = idx
            break

    text = text[start:]

    words = text.split()

    return " ".join(words[:50])

evaluation_df = cases_df[
    ["case_id","case_description"]
].copy()

evaluation_df["query"] = evaluation_df["case_description"].apply(generate_query)

evaluation_df.rename(
    columns={"case_id":"ground_truth"},
    inplace=True
)

evaluation_df.head()

,ground_truth,case_description,query
0,case_001,nama lengkap : muhammad faris isra hatala bin ...,bahwa terdakwa diajukan ke persidangan oleh pe...
1,case_002,a h agung blik indonesi p u t u s a n nomor : ...,bahwa terdakwa diajukan kepersidangan oleh pen...
2,case_003,a h agung blik indonesi p u t u s a n nomor : ...,bahwa terdakwa diajukan kepersidangan oleh pen...
3,case_004,a h agung blik indonesi p u t u s a n nomor 13...,menimbang bahwa berdasarkan alat bukti barang ...
4,case_005,a h agung blik indonesi p u t u s a n nomor 13...,menimbang bahwa berdasarkan alat bukti barang ...


# Evaluation Function

In [102]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

def evaluate_model(retrieve_function):

    y_true=[]
    y_pred=[]

    for _,row in evaluation_df.iterrows():

        result = retrieve_function(
            row["query"],
            top_k=1
        )

        y_true.append(row["ground_truth"])
        y_pred.append(result.iloc[0]["case_id"])

    return {
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,average="macro",zero_division=0),
        "Recall":recall_score(y_true,y_pred,average="macro",zero_division=0),
        "F1-Score":f1_score(y_true,y_pred,average="macro",zero_division=0)
    }

# Evaluate TF-IDF

In [103]:
tfidf_metrics = evaluate_model(
    retrieve_tfidf
)

tfidf_metrics

{'Accuracy': 0.66,
 'Precision': 0.5578571428571428,
 'Recall': 0.66,
 'F1-Score': 0.5863333333333334}

# Evaluate IndoBERT

In [104]:
indobert_metrics = evaluate_model(
    retrieve_indobert
)

indobert_metrics

{'Accuracy': 0.14,
 'Precision': 0.07033333333333333,
 'Recall': 0.14,
 'F1-Score': 0.08496969696969697}

# Comparison Metrics

In [105]:
comparison = pd.DataFrame([
    {
        "Model":"TF-IDF",
        **tfidf_metrics
    },
    {
        "Model":"IndoBERT",
        **indobert_metrics
    }
])

comparison

,Model,Accuracy,Precision,Recall,F1-Score
0,TF-IDF,0.66,0.557857,0.66,0.586333
1,IndoBERT,0.14,0.070333,0.14,0.084970


# Simpan Comparison Metrics

In [106]:
comparison.to_csv(
    "data/results/comparison_metrics.csv",
    index=False
)

comparison

,Model,Accuracy,Precision,Recall,F1-Score
0,TF-IDF,0.66,0.557857,0.66,0.586333
1,IndoBERT,0.14,0.070333,0.14,0.084970
